# ML Fatigue Model Training & Evaluation

This notebook implements the training pipeline for the **Multimodal Driver Fatigue Detection System** using a 19-dimensional feature vector and a calibrated LightGBM classifier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import joblib
import os
import sys
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay

# Add project root to path for feature_schema
sys.path.append('..')
from backend.ml.feature_schema import FEATURE_ORDER

pd.set_option('display.max_columns', None)

## 1. Load Dataset
Loading the synthetic dataset generated for the hackathon.

In [ ]:
DATA_PATH = "../data/training_data.csv"
df = pd.read_csv(DATA_PATH)

print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Dataset Statistics & EDA

In [ ]:
# Label Distribution
sns.countplot(x='fatigue_label', data=df)
plt.title("Class Distribution (0=Alert, 1=Fatigue)")
plt.show()

In [ ]:
# Feature Distributions for EAR and MAR
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(data=df, x='EAR_mean', hue='fatigue_label', kde=True, ax=axes[0])
sns.histplot(data=df, x='MAR_max', hue='fatigue_label', kde=True, ax=axes[1])
plt.show()

## 3. Train/Test Split

In [ ]:
X = df[FEATURE_ORDER]
y = df['fatigue_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

## 4. Model Training & Calibration
Following PRD hyperparameters and Platt scaling.

In [ ]:
LGBM_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "binary",
    "random_state": 42,
    "verbosity": -1
}

base_model = lgb.LGBMClassifier(**LGBM_PARAMS)
calibrated_model = CalibratedClassifierCV(estimator=base_model, method='sigmoid', cv=5)

calibrated_model.fit(X_train, y_train)

## 5. Evaluation

In [ ]:
y_pred = calibrated_model.predict(X_test)
y_prob = calibrated_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# ROC Curve
RocCurveDisplay.from_estimator(calibrated_model, X_test, y_test)
plt.title("Receiver Operating Characteristic (ROC)")
plt.show()

## 6. Feature Importance

In [ ]:
# Retrieve base LightGBM feature importance from the first fold of CalibratedClassifierCV
importance = calibrated_model.calibrated_classifiers_[0].estimator.feature_importances_
feat_imp = pd.Series(importance, index=FEATURE_ORDER).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x=feat_imp.values, y=feat_imp.index)
plt.title("LightGBM Feature Importance (Mean Gain/Split)")
plt.show()

## 7. Model Export

In [ ]:
MODEL_EXPORT_PATH = "../backend/ml/model/fatigue_model.pkl"
if not os.path.exists("../backend/ml/model"): 
    os.makedirs("../backend/ml/model")
    
joblib.dump(calibrated_model, MODEL_EXPORT_PATH)
print(f"Model exported successfully to {MODEL_EXPORT_PATH}")